# 2. Convolutional Neural Networks (CNNs)

CNNs exploit spatial structure in images through local connectivity and weight sharing. This notebook covers:
- **Convolution** and **pooling** operations
- Building a CNN for **CIFAR-10** classification
- Feature map visualization
- Training and evaluation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

%matplotlib inline
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2.1 Convolution Operation

A 2D convolution slides a kernel $K$ over the input $I$:

$$(I * K)[i, j] = \sum_m \sum_n I[i+m, j+n] \cdot K[m, n]$$

Key parameters: **kernel size**, **stride**, **padding**, **number of filters**.

In [ ]:
# Manual convolution demonstration
from scipy.signal import convolve2d

# Create a simple image with edges
img = np.zeros((8, 8))
img[2:6, 2:6] = 1.0

# Edge detection kernels
kernel_h = np.array([[-1, -1, -1],
                     [ 0,  0,  0],
                     [ 1,  1,  1]])  # Horizontal edges

kernel_v = np.array([[-1, 0, 1],
                     [-1, 0, 1],
                     [-1, 0, 1]])  # Vertical edges

out_h = convolve2d(img, kernel_h, mode='same')
out_v = convolve2d(img, kernel_v, mode='same')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Input Image')
axes[1].imshow(out_h, cmap='RdBu')
axes[1].set_title('Horizontal Edge Filter')
axes[2].imshow(out_v, cmap='RdBu')
axes[2].set_title('Vertical Edge Filter')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2.2 CIFAR-10 Dataset

CIFAR-10 contains 60,000 color images (32x32x3) in 10 classes.

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

train_data = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_data = datasets.CIFAR10(root='./data', train=False, transform=transform_test)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=256)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

# Show samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
raw_data = datasets.CIFAR10(root='./data', train=True, transform=transforms.ToTensor())
for i, ax in enumerate(axes.flat):
    img, label = raw_data[i]
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(classes[label])
    ax.axis('off')
plt.suptitle('CIFAR-10 Samples')
plt.tight_layout()
plt.show()

## 2.3 CNN Architecture

Our CNN follows a classic pattern:
- **Conv -> BatchNorm -> ReLU -> MaxPool** (repeated)
- **Fully connected** layers for classification

Output size after convolution: $\lfloor(W - K + 2P)/S\rfloor + 1$

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 3x32x32 -> 32x16x16
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 2: 32x16x16 -> 64x8x8
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 3: 64x8x8 -> 128x4x4
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

In [ ]:
# Training
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
n_epochs = 10
train_losses, test_accs = [], []

for epoch in range(n_epochs):
    model.train()
    running_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # Evaluate
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            correct += (model(images).argmax(1) == labels).sum().item()
    
    avg_loss = running_loss / len(train_loader)
    acc = correct / len(test_data)
    train_losses.append(avg_loss)
    test_accs.append(acc)
    print(f"Epoch {epoch+1:>2}/{n_epochs} | Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, 'o-')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Train Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)
ax2.plot(test_accs, 'o-', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Test Accuracy')
ax2.set_title('Test Accuracy')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2.4 Feature Map Visualization

Let us see what the first convolutional layer learns.

In [ ]:
# Grab first conv layer filters
filters = model.features[0].weight.data.cpu()

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    if i < filters.shape[0]:
        filt = filters[i]
        # Normalize to [0, 1] for display
        filt = (filt - filt.min()) / (filt.max() - filt.min() + 1e-8)
        ax.imshow(filt.permute(1, 2, 0))
    ax.axis('off')
plt.suptitle('Learned Filters (First Conv Layer)')
plt.tight_layout()
plt.show()

## Key Takeaways

- **Convolution** extracts local features through learned filters
- **Pooling** reduces spatial dimensions and provides translation invariance
- **Batch normalization** stabilizes training; **dropout** prevents overfitting
- Stacking conv blocks creates a hierarchy: edges -> textures -> parts -> objects
- Data augmentation (flips, crops) is essential for good generalization on small datasets